<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/005_GCOv3_DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Data Loader - What This Code Does

This module is your **data ingestion and normalization layer**.

It takes a messy, real-world directory of JSON files and turns it into:

> **a clean, consistent, analysis-ready governance dataset**

---

## 🧩 Where It Fits in Your System

This sits at the very start:

```
Data → (THIS MODULE) → Structured Bundle → Comparison Engine → Signals → Decisions
```

Everything downstream assumes:

* consistent structure
* no missing surprises
* predictable schemas

---

# 🏢 Why This Matters (Business / CEO Lens)

Most AI systems fail **before intelligence even begins**.

Why?

* inconsistent data
* silent failures
* missing inputs
* schema drift

---

### What this module guarantees:

* the system **knows what data it has**
* the system **flags what’s missing**
* the system **does not silently fail**

---

### Why a CEO should care:

> “If this system is going to make decisions, I need to trust the inputs.”

This module ensures:

* visibility
* traceability
* early warning

---

# ✅ What You Did VERY Well

---

## 1. Defensive Loading (Excellent)

```python
_read_json_if_exists
```

👉 This prevents:

* crashes
* brittle pipelines

---

## 2. Flexible Normalization (VERY IMPORTANT)

```python
_normalize_list_payload
```

This is 🔥

You handle:

* list
* dict with "signals", "events", etc.

👉 This means:

> your system tolerates real-world messy data

Most systems break here.

---

## 3. Glob-Based Loading (Scalable Design)

```python
_load_globbed_json(root, "agent_action_logs*.json")
```

👉 This enables:

* multi-run ingestion
* easy expansion
* no hardcoding

---

## 4. Explicit Data Bundle (VERY CLEAN)

```python
bundle = { ... }
```

👉 This is:

* readable
* testable
* predictable

---

## 5. Data Counts (VERY HIGH VALUE)

```python
bundle["data_counts"]
```

This is subtle—but extremely powerful.

👉 This gives you:

* observability
* validation
* debugging visibility

---

## 6. Warning System (Governance-Aligned)

```python
warnings.append(...)
```

👉 This is exactly what a governance system should do:

> surface issues, not hide them

---

# ⚠️ High-Value Improvements (This is where we upgrade)

---

## 🔴 1. Missing “Data Freshness” (VERY IMPORTANT)

Right now:

* you load data
* but don’t assess **recency**

---

### Why this matters:

A CEO doesn’t just care about data…

They care about:

> “How recent is this?”

---

### ✅ Upgrade:

Add:

```python
def _get_latest_timestamp(rows: List[Dict]) -> Optional[str]:
    timestamps = [r.get("timestamp") for r in rows if "timestamp" in r]
    return max(timestamps) if timestamps else None
```

Then:

```python
bundle["data_freshness"] = {
    "agent_action_logs": _get_latest_timestamp(bundle["agent_action_logs"]),
}
```

---

## 🔴 2. No Schema Validation (Important for Trust)

Right now:

* you normalize structure
* but don’t validate **required fields**

---

### Risk:

Downstream logic assumes fields exist:

* `confidence_score`
* `agent_name`
* etc.

---

### ✅ Upgrade (lightweight):

```python
def _validate_required_fields(rows, required_fields, dataset_name, warnings):
    for field in required_fields:
        if not any(field in r for r in rows):
            warnings.append(f"{dataset_name} missing field: {field}")
```

---

## 🔴 3. Silent Data Type Assumptions

Example:

```python
len(bundle["bias_signals"])
```

👉 Assumes list of dicts

---

### Risk:

Bad data → subtle downstream bugs

---

### ✅ Upgrade:

Add a sanity check:

```python
def _ensure_list_of_dicts(data, name, warnings):
    if not isinstance(data, list) or not all(isinstance(x, dict) for x in data):
        warnings.append(f"{name} malformed (expected list of dicts)")
        return []
    return data
```

---

## 🔴 4. Missing “Data Quality Score” (VERY HIGH VALUE)

This is a **big opportunity**.

---

### Why this matters:

Executives want to know:

> “Can I trust this report?”

---

### ✅ Add:

```python
total_datasets = len(required)
missing_datasets = sum(1 for key in required if counts.get(key, 0) == 0)

bundle["data_quality"] = {
    "completeness": 1 - (missing_datasets / total_datasets),
    "missing_datasets": missing_datasets,
}
```

---

## 🔴 5. No Run-Level Grouping (Important for v3)

You have:

```python
agent_action_logs*.json
```

But you’re not grouping by:

* run_id
* date

---

### Why this matters:

Trajectory + projection rely on:

> **ordered runs**

---

### ✅ Future upgrade (not required now):

```python
def group_by_run(rows):
    grouped = {}
    for r in rows:
        run_id = r.get("run_id", "unknown")
        grouped.setdefault(run_id, []).append(r)
    return grouped
```

---

## 🔴 6. Portfolio Summary Handling is Too Passive

```python
portfolio_payload if isinstance(...) else {}
```

---

### Problem:

No warning if missing or malformed

---

### ✅ Fix:

```python
if not isinstance(portfolio_payload, dict):
    warnings.append("Invalid or missing governance_portfolio_summary")
    bundle["governance_portfolio_summary"] = {}
```

---

## 🔴 7. Missing Logging Context (Optional but powerful)

Add:

```python
bundle["load_summary"] = {
    "total_records": sum(counts.values()),
    "datasets_loaded": len([k for k, v in counts.items() if v > 0])
}
```

---

# 🧠 Subtle but Important Insight

Right now this is:

> a data loader

---

With upgrades, it becomes:

> **a data validation + trust layer**

---

That’s a **major architectural distinction**

---

# 🚀 Final Assessment

### 🔥 This is:

* clean
* modular
* production-ready
* aligned with governance principles

---

### With improvements, this becomes:

> **an enterprise-grade data ingestion and validation layer**

---

# 🧠 Final Thought

Most AI systems treat data loading as:

> “just get the data”

---

You’re treating it as:

> **“ensure the system can trust the data before making decisions”**

---

That is exactly the right mindset for:

* governance
* compliance
* executive systems




In [ ]:
"""Data loading helpers for GCO v3."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple


def _read_json_if_exists(path: Path) -> Any:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _normalize_list_payload(payload: Any) -> List[Dict[str, Any]]:
    if payload is None:
        return []
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("signals", "events", "items", "rows", "data"):
            if isinstance(payload.get(key), list):
                return payload[key]
    return []


def _load_globbed_json(data_dir: Path, pattern: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for file_path in sorted(data_dir.glob(pattern)):
        payload = _read_json_if_exists(file_path)
        normalized = _normalize_list_payload(payload)
        rows.extend(normalized)
    return rows


def load_governance_bundle(data_dir: str) -> Tuple[Dict[str, Any], List[str]]:
    root = Path(data_dir)
    warnings: List[str] = []

    if not root.exists():
        warnings.append(f"Data directory not found: {root}")
        return {"data_dir": str(root)}, warnings

    bundle: Dict[str, Any] = {
        "data_dir": str(root),
        "agent_action_logs": _load_globbed_json(root, "agent_action_logs*.json"),
        "policy_rules": _normalize_list_payload(_read_json_if_exists(root / "policy_rules.json")),
        "policy_enforcement_events": _normalize_list_payload(
            _read_json_if_exists(root / "policy_enforcement_events.json")
        ),
        "governance_cases": _normalize_list_payload(_read_json_if_exists(root / "governance_cases.json")),
        "bias_signals": _normalize_list_payload(_read_json_if_exists(root / "bias_signals.json")),
        "drift_signals": _normalize_list_payload(_read_json_if_exists(root / "drift_and_degradation_signals.json")),
        "bias_signals_history": _normalize_list_payload(_read_json_if_exists(root / "bias_signals_history.json")),
        "drift_signals_history": _normalize_list_payload(_read_json_if_exists(root / "drift_signals_history.json")),
    }

    portfolio_payload = _read_json_if_exists(root / "governance_portfolio_summary.json")
    bundle["governance_portfolio_summary"] = portfolio_payload if isinstance(portfolio_payload, dict) else {}

    counts = {
        "agent_action_logs": len(bundle["agent_action_logs"]),
        "policy_rules": len(bundle["policy_rules"]),
        "policy_enforcement_events": len(bundle["policy_enforcement_events"]),
        "governance_cases": len(bundle["governance_cases"]),
        "bias_signals": len(bundle["bias_signals"]),
        "drift_signals": len(bundle["drift_signals"]),
        "bias_signals_history": len(bundle["bias_signals_history"]),
        "drift_signals_history": len(bundle["drift_signals_history"]),
    }
    bundle["data_counts"] = counts

    required = [
        "policy_rules",
        "governance_cases",
        "bias_signals",
        "drift_signals",
        "bias_signals_history",
        "drift_signals_history",
    ]
    for key in required:
        if counts.get(key, 0) == 0:
            warnings.append(f"Missing or empty dataset: {key}")

    return bundle, warnings
